## Predicting Leaks
In this notebook, I will be feature engineering some variables that will help us identify each pipe's likelihood of leaking.

#### Process:
- Joined pipes.csv with train.csv on pipe_id
- Created 'Age' variable for each pipe using 'Lay date' from pipes.csv and 'Date' from train.csv
- Created 'Length' variable for each pipe using Euclidean distance
- Created target variable 'Leaked' (0 for not leaked, 1 for leaked)
- Created 'is_iron' variable to distinguish whether a pipe needs to be replaced or not


### Libraries and Reading in Data

In [1]:
import os
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import cKDTree

In [2]:
pipes = pd.read_csv("../data/raw/pipes.csv", parse_dates=["Lay date"])
leaks = pd.read_csv("../data/raw/train.csv", parse_dates=["Date"])

### Age in years
15 pipes have no lay date, so their age is unknown, therefore I have decided to drop these pipes.

In [3]:
df = pipes.merge(leaks, on="Pipe ID", how="left")

df["Leaked"] = df["Date"].notna().astype(int)
df["Date"] = df["Date"].fillna(pd.Timestamp.today().normalize())
df["Age"] = (df["Date"] - df["Lay date"]).dt.days / 365.25  # years

df = df.dropna(subset=["Lay date"]).reset_index(drop=True)
df['Age'].describe()

count    43024.000000
mean        67.516705
std          8.842732
min         34.242300
25%         62.335387
50%         64.651608
75%         74.973306
max         81.771389
Name: Age, dtype: float64

### Pipe length

Euclidean distance between each pipe's two endpoints, in the same units as the GPS coordinates.

In [8]:
df["Length"] = np.sqrt((df["GPS x2"] - df["GPS x1"])**2 + (df["GPS y2"] - df["GPS y1"])**2)
df['Length'].describe()

count    43016.000000
mean        17.742526
std         15.770646
min          0.170000
25%          7.985775
50%         11.312891
75%         20.685710
max         74.881445
Name: Length, dtype: float64

### Finding which pipes are required to be replaced
8 pipes have no material recorded, but after further investigation, it seems these pipes are already polyurathane, (extrapolated based on their lay dates and surface). Therefore I have decided to drop these pipes. 

In [5]:
df = df.dropna(subset=['Material'])
df['is_iron'] = df['Material'].str.contains(r'iron$', regex=True).astype("Int64")
df['is_iron'].value_counts()

is_iron
1    38440
0     4576
Name: count, dtype: Int64

### Looking at lengths of those needing replacement

In [ ]:
df[df["is_iron"] == 1]["Length"].sort_values(ascending=False)


6463     74.881445
6462     74.873639
5692     74.712024
7295     74.578169
7155     74.568536
           ...    
10065     0.320156
11043     0.313209
9887      0.286531
15045     0.268328
14321     0.170000
Name: Length, Length: 38440, dtype: float64

In [7]:
df

,Pipe ID,Lay date,Material,Surface,GPS x1,GPS y1,GPS x2,GPS y2,Date,Time,Cost,Leaked,Age,Length,is_iron
0,P01001,1955-03-11,copper,grassland,6136.29,5020.97,6124.15,5020.97,2026-10-08,NaN,NaN,0,71.578371,12.140000,0
1,P01002,1971-09-19,cast iron,water,6512.35,4147.58,6536.59,4201.94,2023-06-21,04:13,3331.32,1,51.753593,59.519637,1
2,P01003,1971-09-18,cast iron,water,6536.59,4201.94,6518.42,4201.93,2024-04-28,21:05,4000.21,1,52.610541,18.170003,1
3,P01004,1982-05-22,cast iron,structure,2836.68,3377.95,2848.80,3405.12,2026-10-08,NaN,NaN,0,44.380561,29.750686,1
4,P01005,1971-07-03,cast iron,water,5117.28,4272.91,5126.37,4297.16,2024-07-03,10:05,2663.18,1,53.002053,25.897695,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43019,P44035,1971-07-12,cast iron,water,4791.94,4288.78,4797.85,4334.63,2026-10-08,NaN,NaN,0,55.241615,46.229326,1
43020,P44036,1971-07-05,cast iron,water,4957.39,4288.78,4951.48,4334.63,2023-09-22,11:28,2230.15,1,52.216290,46.229326,1
43021,P44037,1971-07-06,cast iron,water,4957.39,4288.78,4963.30,4242.93,2026-10-08,NaN,NaN,0,55.258042,46.229326,1
43022,P44038,1971-07-11,cast iron,water,4957.39,4105.39,4963.30,4151.24,2026-10-08,NaN,NaN,0,55.244353,46.229326,1


## Next steps
1. Test different models for prediction to see which suits our data the best.
2. Do more research on time series data because I believe this is very much a time series problem. (Unfortunately, I took STAT445 instead of STAT485, so I'm not sure what methods are required with time series data)
3. Maybe look into building a random forest to try and predict likelihood of leaking to identify at risk areas to target first with our work units?